# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a walkthrough for exploring the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, following best practices for referencing dataset entities by their `@id` fields as defined by the [Croissant schema](https://mlcommons.github.io/croissant/).

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

The dataset includes multiple record sets, fields, and detailed metadata associated with ordered logistic regression analyses on knowledge adoption in rangeland management households (Northern Kenya).

In [ ]:
# Make sure mlcroissant is installed (uncomment for first time use)
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata using the Croissant schema URL. You may need to restart the kernel after installing the package.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a single metadata object
print('Dataset Title:', metadata.name)
print('Description:', metadata.description)
print('Identifier:', metadata.identifier)
print('Date Published:', metadata.datePublished)
print('Coverage:', metadata.spatialCoverage)
print('\n--- Data Collection Note ---\n' + (metadata.dataCollection if hasattr(metadata, 'dataCollection') else ''))

## 2. Data Overview
List available record sets and their IDs, then inspect fields for each one. All entities are referenced by their `@id`.

In [ ]:
# List all record sets present in the dataset, referencing by @id.
record_sets_info = dataset.record_sets  # a list of RecordSet objects

print('Available record sets:')
for rs in record_sets_info:
    print(f"  - @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")

# For illustration, print fields in each record set
print('\nFields in each record set:')
for rs in record_sets_info:
    fields = rs.get('field', [])
    print(f"\nRecord set @id: {rs['@id']}")
    if isinstance(fields, list):
        for field in fields:
            print(f"  - Field @id: {field.get('@id', str(field))} | name: {field.get('name', 'N/A')} | type: {field.get('dataType', 'N/A')}")
    elif isinstance(fields, dict):
        print(f"  - Field @id: {fields.get('@id', str(fields))} | name: {fields.get('name', 'N/A')} | type: {fields.get('dataType', 'N/A')}")
    else:
        print("  (No fields listed)")

## 3. Data Extraction
Load all records from each record set (chosen by their `@id`) into DataFrames.

**Note:** Replace the list of record set `@id`s below if you want to restrict to just a subset. Each record set is referenced by its `@id`.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print('Processing record sets:', record_set_ids)

dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records for record set {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records; columns: {df.columns.tolist()}")
    else:
        print("No records found.")

# For demonstration, print the first few rows of the first available record set with data
if dataframes:
    example_rs_id = next(iter(dataframes))
    print(f"\nColumns in record set {example_rs_id}:\n", dataframes[example_rs_id].columns.tolist())
    display(dataframes[example_rs_id].head())
else:
    print("No data was loaded from the record sets.")

## 4. Exploratory Data Analysis (EDA)
Process and explore one numeric field using its column and field `@id`. Typical steps include filtering, normalization, and grouping.

> **Hint:** Replace `<numeric_field_id>` and `<group_field_id>` with actual column/field ids from section 2 above. The records from each record set are columns in the associated DataFrame, and should be referenced by the column's `@id`.

In [ ]:
# Example setup: select a record set and field for processing.
# (You may need to inspect print outputs above to find the correct ids)

# Choose a record set with numerical data (replace with actual @id as needed):
example_record_set_id = next(iter(dataframes)) if dataframes else None
df = dataframes.get(example_record_set_id)

if df is not None:
    print(f"Inspecting columns for numeric fields in {example_record_set_id}:")
    print(df.dtypes)

    # Attempt to auto-select a likely numeric field (or set below manually)
    possible_numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if possible_numeric_cols:
        numeric_field_id = possible_numeric_cols[0]
    else:
        numeric_field_id = df.columns[0]  # fallback option

    print(f"Selected numeric field: {numeric_field_id}")
    
    # Example: filter for values above a threshold
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records")

    # Normalize selected column
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field (take the first object column)
    group_candidates = [col for col in df.columns if df[col].dtype == 'object']
    if group_candidates:
        group_field_id = group_candidates[0]
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count'])
        print(f"\nGrouping by {group_field_id}:")
        print(grouped.head())
    else:
        print("No categorical field detected for grouping.")
else:
    print("No DataFrame is available to analyze.")

## 5. Visualization
Visualize the data distribution or field relationships using matplotlib or seaborn.

> **Note:** Replace variables below as appropriate for your selected `record_set_id` and fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id in df:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    # Scatter/grouped boxplot visualization if grouping field exists
    if 'group_field_id' in locals() and group_field_id in df:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=60)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization. Please make sure you have loaded the DataFrame and specified the field IDs.")

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore a Croissant-formatted dataset with the `mlcroissant` library. We covered how to:
- Load dataset metadata and describe high-level characteristics.
- List record sets and examine their fields using their `@id` references.
- Load records into DataFrames, process and analyze a selected numeric field, and group by a categorical field.
- Visualize distributions and group differences.

This approach ensures all data manipulations remain consistent with the schema, and referencing by `@id` supports reproducibility and clarity for downstream analysis.

> For further analysis, explore other record sets and fields by leveraging their `@id` in code.